# GEMASTIK XIX 2026 - DIVISI III PENAMBANGAN DATA
## MASTER NOTEBOOK: TASK 2 - TOPIC MODELING (UNSUPERVISED LEARNING)
---
**Deskripsi Notebook:**
Notebook ini dirancang khusus untuk memecahkan persoalan **Topic Modeling** secara *unsupervised* dalam kondisi **100% OFFLINE** pada PC Panitia (Intel Core i7-12700, RAM 32GB).

### Alur Kerja (Pipeline):
1. **Pemuatan Korpus & Normalisasi Teks Khusus Pemodelan Topik**
2. **Vektorisasi Teks (TF-IDF vs Bag-of-Words)**
3. **Metode Utama: Non-Negative Matrix Factorization (NMF)**
   - *Keunggulan*: Sangat cepat di CPU i7-12700, topik koheren dan non-overlapping pada dokumen teks pendek-menengah.
4. **Metode Pembanding: Latent Dirichlet Allocation (LDA)**
   - Memberikan justifikasi probabilitas Bayesien $P(w|z)$ dan $P(z|d)$.
5. **Interpretasi & Pembuatan Nama Topik (Human-Readable Topic Labeling)**
6. **Visualisasi Komprehensif (Bar Chart Kata Topik, Document-Topic Heatmap, Scatter Proyeksi 2D)**
7. **Ekspor Fitur Matriks Distribusi Topik untuk Dioper ke Task 1 (Integrasi Dua Task!)**



In [ ]:
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import NMF, LatentDirichletAllocation, TruncatedSVD
from sklearn.manifold import TSNE

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
os.makedirs('output_charts', exist_ok=True)
os.makedirs('output_topics', exist_ok=True)
print("Modul Topic Modeling berhasil dimuat 100% offline!")


## 1. Pemuatan & Pembersihan Korpus Teks



In [ ]:
DATA_PATH = "cases/aksi_28_okt/train.csv"  # Ganti sesuai path dataset panitia
TEXT_COL = "clean_text"

if os.path.exists(DATA_PATH):
    df_corpus = pd.read_csv(DATA_PATH)
else:
    # Fallback dummy data
    df_corpus = pd.DataFrame({
        TEXT_COL: [
            "Pemberantasan korupsi dan mafia hukum harus ditegakkan demi keadilan rakyat",
            "Kebijakan tarif pajak dan stabilitas ekonomi nasional perlu diperhatikan pemerintah",
            "Demonstrasi mahasiswa menuntut reformasi undang-undang ketenagakerjaan berlangsung tertib",
            "Peningkatan kualitas pendidikan digital dan teknologi informasi di sekolah daerah 3T",
            "Penanganan bencana alam dan bantuan logistik korban banjir dikirimkan segera"
        ] * 100
    })

STOPWORDS_EXTENDED = {
    'yang', 'untuk', 'pada', 'ke', 'para', 'namun', 'menurut', 'antara', 'dia', 'dua',
    'ia', 'seperti', 'jika', 'sehingga', 'kembali', 'dan', 'ini', 'karena', 'oleh',
    'saat', 'harus', 'sementara', 'setelah', 'belum', 'kami', 'telah', 'ada', 'mereka',
    'sudah', 'bisa', 'akan', 'bukan', 'atau', 'dapat', 'juga', 'dari', 'dalam', 'di',
    'dengan', 'adalah', 'itu', 'tersebut', 'tentang', 'bahwa', 'saja', 'lebih', 'agar',
    'sangat', 'lalu', 'semua', 'hanya', 'banyak', 'bagi', 'terhadap', 'menjadi', 'maka'
}

def clean_for_topic_modeling(text):
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'[-+]?[0-9]+', '', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = [w for w in text.split() if w not in STOPWORDS_EXTENDED and len(w) > 3]
    return " ".join(tokens)

df_corpus['clean_topic_text'] = df_corpus[TEXT_COL].apply(clean_for_topic_modeling)
print(f"Total Dokumen Korpus: {len(df_corpus)}")
display(df_corpus[[TEXT_COL, 'clean_topic_text']].head(3))


## 2. Vektorisasi TF-IDF
Untuk NMF, representasi berbasis **TF-IDF dengan Sublinear TF** memberikan penalti yang optimal pada kata-kata umum dan mengangkat kata kunci unik tema.



In [ ]:
tfidf_vectorizer = TfidfVectorizer(
    max_df=0.85,           # Abaikan kata yang muncul di >85% dokumen (terlalu umum)
    min_df=3,              # Abaikan kata typo yang hanya muncul di <3 dokumen
    max_features=8000,     # Ambil 8,000 fitur terpenting
    ngram_range=(1, 2),    # Unigram dan bigram
    sublinear_tf=True
)

X_tfidf = tfidf_vectorizer.fit_transform(df_corpus['clean_topic_text'])
terms = np.array(tfidf_vectorizer.get_feature_names_out())
print(f"Matriks TF-IDF: {X_tfidf.shape[0]} dokumen x {X_tfidf.shape[1]} fitur kosakata unik")


## 3. Eksekusi NMF (Non-Negative Matrix Factorization)
NMF memfaktorkan matriks dokumen-kata $V pprox W 	imes H$:
- $W$ ($N 	imes K$): Distribusi bobot topik pada masing-masing dokumen.
- $H$ ($K 	imes M$): Bobot kata kunci untuk masing-masing topik.



In [ ]:
N_TOPICS = 5  # Tentukan jumlah topik (biasanya 4 - 8 tergantung domain kasus)

nmf_model = NMF(
    n_components=N_TOPICS,
    init='nndsvda',        # Inisialisasi deterministik & stabil
    random_state=42,
    max_iter=400
)

# W = Bobot Topik per Dokumen | H = Bobot Kata per Topik
doc_topic_matrix = nmf_model.fit_transform(X_tfidf)
topic_word_matrix = nmf_model.components_

print(f"Bentuk Matriks Dokumen-Topik (W) : {doc_topic_matrix.shape}")
print(f"Bentuk Matriks Topik-Kata (H)    : {topic_word_matrix.shape}")


## 4. Ekstraksi Kata Kunci & Pelabelan Topik Human-Readable
Menampilkan tabel ringkasan kata-kata teratas per topik beserta pemberian label tema yang jelas untuk presentasi.



In [ ]:
N_TOP_WORDS = 10
topic_summaries = []

for topic_idx, topic in enumerate(topic_word_matrix):
    top_indices = topic.argsort()[:-N_TOP_WORDS-1:-1]
    top_words = terms[top_indices]
    top_weights = topic[top_indices]
    
    words_formatted = ", ".join([f"{w} ({wt:.2f})" for w, wt in zip(top_words, top_weights)])
    topic_summaries.append({
        "Topic_ID": f"Topik {topic_idx + 1}",
        "Top_Keywords": ", ".join(top_words[:7]),
        "Keywords_with_Weights": words_formatted
    })

df_topic_summary = pd.DataFrame(topic_summaries)
df_topic_summary.to_csv("output_topics/topic_modeling_summary.csv", index=False)
print("TABEL RINGKASAN TOPIK TEMATIK:")
display(df_topic_summary[['Topic_ID', 'Top_Keywords']])


## 5. Visualisasi Bar Chart Kata Kunci per Topik (Untuk Slide PPT)
Grafik horizontal bar chart yang memperlihatkan kata kunci paling dominan pada setiap topik.



In [ ]:
fig, axes = plt.subplots(1, N_TOPICS, figsize=(4 * N_TOPICS, 5), sharey=False)
if N_TOPICS == 1:
    axes = [axes]

colors = sns.color_palette("tab10", N_TOPICS)

for topic_idx, topic in enumerate(topic_word_matrix):
    top_indices = topic.argsort()[:-8:-1]
    top_words = terms[top_indices][::-1]
    top_scores = topic[top_indices][::-1]
    
    axes[topic_idx].barh(top_words, top_scores, color=colors[topic_idx], edgecolor='black', alpha=0.85)
    axes[topic_idx].set_title(f"Topik {topic_idx + 1}", fontsize=12, fontweight='bold')
    axes[topic_idx].set_xlabel("Bobot NMF")

plt.suptitle("Distribusi Kata Kunci Teratas per Topik Tematik (NMF)", fontsize=14, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig("output_charts/nmf_topic_keywords_barchart.png", dpi=200, bbox_inches='tight')
plt.show()
print("[OK] Grafik Bar Chart Topik diekspor ke: output_charts/nmf_topic_keywords_barchart.png")


## 6. Visualisasi Proyeksi 2D Klaster Topik Dokumen (PCA / t-SNE)
Menampilkan peta sebaran dokumen dalam ruang 2 dimensi berdasarkan topik dominannya.



In [ ]:
# Menentukan topik dominan untuk setiap dokumen
dominant_topics = np.argmax(doc_topic_matrix, axis=1)
df_corpus['dominant_topic'] = [f"Topik {t+1}" for t in dominant_topics]

# Proyeksi 2D menggunakan TruncatedSVD (sangat cepat & deterministik di CPU)
svd_2d = TruncatedSVD(n_components=2, random_state=42)
coords_2d = svd_2d.fit_transform(doc_topic_matrix)

plt.figure(figsize=(9, 6))
sns.scatterplot(
    x=coords_2d[:, 0],
    y=coords_2d[:, 1],
    hue=df_corpus['dominant_topic'],
    palette='tab10',
    alpha=0.75,
    s=40
)
plt.title("Proyeksi 2D Klaster Dokumen Berdasarkan Topik Dominan (LSA / SVD)", fontsize=13, fontweight='bold')
plt.xlabel("Dimensi Semantik 1")
plt.ylabel("Dimensi Semantik 2")
plt.legend(title="Topik Dominan", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig("output_charts/topic_clusters_projection_2d.png", dpi=200, bbox_inches='tight')
plt.show()
print("[OK] Visualisasi Klaster 2D diekspor ke: output_charts/topic_clusters_projection_2d.png")


## 7. Ekspor Fitur Distribusi Topik untuk Dioper ke Task 1 (Integrasi Dua Task)
Nilai jual utama ke juri: Matriks distribusi probabilitas topik dokumen ini ($W$) disimpan ke file CSV untuk di-merge sebagai fitur prediktif tambahan ke Task 1 Klasifikasi!



In [ ]:
topic_col_names = [f"topic_feature_{i+1}" for i in range(N_TOPICS)]
df_topic_features = pd.DataFrame(doc_topic_matrix, columns=topic_col_names)

# Gabungkan dengan ID jika ada
if 'id' in df_corpus.columns:
    df_topic_features['id'] = df_corpus['id']

output_topic_feature_path = "output_topics/topic_features_for_classification.csv"
df_topic_features.to_csv(output_topic_feature_path, index=False)
print("============================================================")
print(f" FITUR TOPIK UNTUK TASK 1 KLASIFIKASI BERHASIL DISIMPAN!")
print("============================================================")
print(f"Lokasi File : {output_topic_feature_path}")
print(f"Dimensi     : {df_topic_features.shape[0]} dokumen x {df_topic_features.shape[1]} fitur topik")
display(df_topic_features.head(3))
